In [ ]:
pip install pruna pillow

In [ ]:
from huggingface_hub import login

login()

In [ ]:
from pruna import PrunaModel, smash, SmashConfig

# Load base model
base_flux = PrunaModel.from_pretrained(
    "black-forest-labs/FLUX.2-klein-4B"
)

# Smash configuration (adjust as needed)
smash_config = SmashConfig({
    # Diffusers Int8: BitsAndBytes 8-bit quantization for transformer
    "diffusers_int8": {
        "diffusers_int8_weight_bits": 8,
        "diffusers_int8_double_quant": False,  # No double quant for 4B (overkill)
        "diffusers_int8_enable_fp32_cpu_offload": False,  # Keep on GPU
        "diffusers_int8_quant_type": "fp4",  # FP4 for better quality
        "diffusers_int8_target_modules": {
            "include": ["*transformer*", "*attn*"],  # Quantize attention + transformer
            "exclude": ["*norm*", "*embed*", "*pe_*"]  # Skip norms, embeddings, positional encoding
        },
    },
    # Torch Compile: JIT compilation for optimized kernels
    "torch_compile": {
        "torch_compile_mode": "reduce-overhead",
        "torch_compile_backend": "inductor",
        "torch_compile_fullgraph": False,
        "torch_compile_dynamic": False,
        "torch_compile_make_portable": True,
        "torch_compile_target": "module_list",
    },
})

# Apply smash
flux = smash(base_flux, smash_config)

# Optional: save optimized artifact
flux.save_pretrained("flux.2_klein_4b_smashed")

In [ ]:
import dspy
from dspy import settings

settings.configure(
    lm=dspy.HFModel(
        model="HuggingFaceTB/SmolVLM-Instruct",
        temperature=0.0,
        top_p=1.0,
        max_tokens=256,
        num_beams=1,
        device="cuda",
        batch_size=4,
    )
)

In [ ]:
class ImageGradeSignature(dspy.Signature):
    """Grade generated image against prompt on multiple dimensions"""
    
    prompt: str = dspy.InputField(
        desc="The original generation prompt"
    )
    image: object = dspy.InputField(
        desc="PIL Image to evaluate"
    )
    
    prompt_adherence: float = dspy.OutputField(
        desc="Score 0-10: How well does image match prompt?"
    )
    aesthetic_quality: float = dspy.OutputField(
        desc="Score 0-10: Visual quality, composition, colors"
    )
    text_correctness: float = dspy.OutputField(
        desc="Score 0-10: Accuracy of text rendering (if any)"
    )
    brand_alignment: float = dspy.OutputField(
        desc="Score 0-10: Professional appearance, on-brand feeling"
    )

class ImageGrader(dspy.Module):
    """DSPy module for VLM-based image grading"""
    
    def __init__(self):
        super().__init__()
        self.grade = dspy.Predict(ImageGradeSignature)
    
    def forward(self, prompt: str, image) -> dict:
        """Grade single image"""
        prediction = self.grade(prompt=prompt, image=image)
        return {
            "prompt_adherence": float(prediction.prompt_adherence),
            "aesthetic_quality": float(prediction.aesthetic_quality),
            "text_correctness": float(prediction.text_correctness),
            "brand_alignment": float(prediction.brand_alignment),
        }

vlm_grader = ImageGrader()

print("✓ VLM grader module created")

In [ ]:
class PromptOptimizationSignature(dspy.Signature):
    """Optimize subject into FLUX-ready prompt using in-context learning"""
    
    subject: str = dspy.InputField(
        desc="High-level subject (e.g., 'Minimalist coffee brand poster')"
    )
    optimized_prompt: str = dspy.OutputField(
        desc="""Detailed FLUX.2-klein-4B generation prompt including:
        - Photography style and type
        - Lighting and atmosphere
        - Composition and framing
        - Color palette and mood
        - Quality/resolution hints
        - Length: 150-250 words
        - NO conflicting instructions"""
    )

class PromptOptimizer(dspy.Module):
    """DSPy module for prompt optimization via few-shot in-context learning"""
    
    def __init__(self):
        super().__init__()
        self.generate = dspy.ChainOfThought(PromptOptimizationSignature)
    
    def forward(self, subject: str) -> str:
        """Optimize subject into detailed prompt"""
        result = self.generate(subject=subject)
        return result.optimized_prompt

prompt_optimizer = PromptOptimizer()

print("✓ Prompt optimizer module created")

In [ ]:
class ScoringWeights:
    """Weights for composite score - emphasize prompt adherence + aesthetics"""
    PROMPT_ADHERENCE = 0.40
    AESTHETIC_QUALITY = 0.35
    TEXT_CORRECTNESS = 0.15
    BRAND_ALIGNMENT = 0.10

def compute_composite_score(scores: dict) -> float:
    """Weighted composite score (0-10 scale)"""
    return (
        ScoringWeights.PROMPT_ADHERENCE * scores.get("prompt_adherence", 0) +
        ScoringWeights.AESTHETIC_QUALITY * scores.get("aesthetic_quality", 0) +
        ScoringWeights.TEXT_CORRECTNESS * scores.get("text_correctness", 0) +
        ScoringWeights.BRAND_ALIGNMENT * scores.get("brand_alignment", 0)
    )

def evaluation_metric(example, pred, trace=None):
    """
    Metric for DSPy MIPROv2 optimization.
    
    NOTE: In this architecture, we use this to potentially optimize
    prompt generation itself. For now, it's a simple scorer.
    """
    if hasattr(pred, 'composite_score'):
        return pred.composite_score / 10.0  # Normalize to [0, 1]
    return 0.5  # Default fallback

print("✓ Scoring metric defined")

In [ ]:
from dspy.teleprompt import MIPROv2

print("Initializing DSPy MIPROv2 optimizer for prompt refinement...")

optimizer = MIPROv2(
    metric=evaluation_metric,
    num_threads=4,
    num_samples=5,  # Few iterations (expensive to generate images)
    auto="medium"   # Auto-configure search space
)

print("✓ MIPROv2 optimizer ready (optional for iterative refinement)")

In [ ]:
SUBJECTS = [
    "Minimalist coffee brand poster with clean typography and steam",
    "Modern eco-friendly packaging design for luxury chocolate",
    "Contemporary tech startup office entrance with glass and wood",
    "Artisanal bakery storefront with warm lighting and fresh bread display",
    "Sustainable fashion brand lookbook with neutral tones and natural fabrics",
    "Boutique hotel lobby with marble and soft ambient lighting",
    "Organic skincare product flat lay with natural ingredients",
    "Furniture showroom displaying minimalist wooden pieces",
    "Plant-based restaurant menu photography with vibrant ingredients",
    "Luxury watch brand catalog shot with professional photography",
]

# For quick testing, use subset
SUBJECTS_TEST = SUBJECTS[:3]

print(f"Total subjects: {len(SUBJECTS)}")
print(f"Test subjects: {len(SUBJECTS_TEST)}")

In [ ]:
from PIL import Image
from pathlib import Path
import json
from tqdm import tqdm

output_dir = Path("./flux_dataset_output")
output_dir.mkdir(exist_ok=True)
images_dir = output_dir / "images"
images_dir.mkdir(exist_ok=True)

def generate_and_grade_candidates(
    subject: str,
    num_candidates: int = 4,
    num_inference_steps: int = 20,
) -> list:
    """
    For a single subject:
    1. Optimize prompt with DSPy
    2. Generate N candidate images with FLUX
    3. Grade each with SmolVLM
    4. Return sorted by composite score
    """
    
    print(f"\n{'='*70}")
    print(f"Subject: {subject}")
    print(f"{'='*70}")
    
    # Step 1: Optimize prompt
    print("  Optimizing prompt...")
    try:
        optimized_prompt = prompt_optimizer(subject=subject)
        print(f"  ✓ Prompt optimized")
    except Exception as e:
        print(f"  ✗ Prompt optimization failed: {e}")
        # Fallback: use subject as-is with enrichment
        optimized_prompt = f"{subject}, professional photography, studio lighting, 4K, sharp focus, vibrant colors, magazine-quality"
    
    print(f"  Prompt: {optimized_prompt[:100]}...")
    
    candidates = []
    
    # Step 2: Generate candidates
    for i in range(num_candidates):
        print(f"  Generating candidate {i+1}/{num_candidates}...", end=" ")
        
        try:
            # Generate with FLUX
            output = flux_smashed(
                prompt=optimized_prompt,
                num_inference_steps=num_inference_steps,
                guidance_scale=3.5,
                height=768,
                width=768,
                seed=42 + i  # Deterministic
            )
            image = output.images[0]
            print("✓", end=" ")
            
            # Step 3: Grade with SmolVLM
            print("(grading...", end=" ")
            scores = vlm_grader(prompt=optimized_prompt, image=image)
            composite = compute_composite_score(scores)
            scores["composite"] = composite
            print("✓)")
            
            # Store result
            candidates.append({
                "image": image,
                "scores": scores,
                "prompt": optimized_prompt,
                "seed": 42 + i
            })
            
            # Print scores
            print(f"    → Adherence: {scores['prompt_adherence']:.1f}, "
                  f"Aesthetic: {scores['aesthetic_quality']:.1f}, "
                  f"Composite: {composite:.2f}")
        
        except Exception as e:
            print(f"✗ Error: {e}")
            continue
    
    if not candidates:
        print("  ✗ No valid candidates generated")
        return []
    
    # Sort by composite score
    candidates.sort(key=lambda x: x["scores"]["composite"], reverse=True)
    
    print(f"  Generated {len(candidates)} valid candidates")
    
    return candidates

In [ ]:
def generate_full_dataset(
    subjects: list,
    candidates_per_subject: int = 4,
    keep_per_subject: int = 2,
    num_inference_steps: int = 20
) -> list:
    """
    Full pipeline:
    1. For each subject → generate candidates → grade → select top K
    2. Save images + metadata
    
    Expected output: len(subjects) × keep_per_subject images
    """
    
    print(f"\n{'#'*70}")
    print(f"DATASET GENERATION PIPELINE")
    print(f"{'#'*70}")
    print(f"Subjects: {len(subjects)}")
    print(f"Candidates/subject: {candidates_per_subject}")
    print(f"Keep/subject: {keep_per_subject}")
    print(f"Expected images: {len(subjects) * keep_per_subject}")
    
    all_records = []
    image_counter = 0
    
    for subject_idx, subject in enumerate(subjects, 1):
        print(f"\n[{subject_idx}/{len(subjects)}]")
        
        # Generate and grade candidates
        candidates = generate_and_grade_candidates(
            subject=subject,
            num_candidates=candidates_per_subject,
            num_inference_steps=num_inference_steps
        )
        
        if not candidates:
            print(f"Skipping subject (no valid candidates)")
            continue
        
        # Keep top K
        selected = candidates[:keep_per_subject]
        
        # Save images and create metadata
        for rank, candidate in enumerate(selected, 1):
            image_id = f"subject_{subject_idx:02d}_rank_{rank}"
            image_path = images_dir / f"{image_id}.png"
            
            # Save image
            candidate["image"].save(str(image_path))
            
            # Create metadata record
            record = {
                "image_id": image_id,
                "subject": subject,
                "optimized_prompt": candidate["prompt"],
                "image_path": str(image_path.relative_to(output_dir)),
                "scores": {
                    "prompt_adherence": float(candidate["scores"]["prompt_adherence"]),
                    "aesthetic_quality": float(candidate["scores"]["aesthetic_quality"]),
                    "text_correctness": float(candidate["scores"]["text_correctness"]),
                    "brand_alignment": float(candidate["scores"]["brand_alignment"]),
                    "composite": float(candidate["scores"]["composite"]),
                },
                "seed": candidate["seed"],
                "rank": rank,
            }
            
            all_records.append(record)
            image_counter += 1
    
    print(f"\n{'#'*70}")
    print(f"✓ PIPELINE COMPLETE: {image_counter} images generated")
    print(f"{'#'*70}\n")
    
    return all_records

In [ ]:
# Test run first (3 subjects, 2 candidates, keep 1)
print("\n⚠️  RUNNING TEST PIPELINE")
test_records = generate_full_dataset(
    subjects=SUBJECTS_TEST,
    candidates_per_subject=2,
    keep_per_subject=1,
    num_inference_steps=20
)

# Save test metadata
metadata_path = output_dir / "metadata_test.json"
with open(metadata_path, "w") as f:
    json.dump(test_records, f, indent=2)

print(f"✓ Test metadata saved to {metadata_path}")

In [ ]:
import datasets

def export_to_hub(records: list, repo_id: str = None):
    """Export dataset to Hugging Face Hub"""
    
    if not repo_id:
        print("Skipping Hub export (no repo_id)")
        return
    
    print(f"Exporting to {repo_id}...")
    
    try:
        hf_dataset = datasets.Dataset.from_dict({
            "image_id": [r["image_id"] for r in records],
            "subject": [r["subject"] for r in records],
            "optimized_prompt": [r["optimized_prompt"] for r in records],
            "composite_score": [r["scores"]["composite"] for r in records],
        })
        
        hf_dataset.push_to_hub(repo_id, private=True)
        print(f"✓ Exported to {repo_id}")
    
    except Exception as e:
        print(f"✗ Export failed: {e}")

# export_to_hub(test_records, repo_id="your-org/flux-optimized-dataset")